# 🏷 Auto-interp — Qwen3.6-27B paper-grade SAEs

Generates semantic labels for the most-active features at L11 / L31 / L55 of [`caiovicentino1/qwen36-27b-sae-papergrade`](https://huggingface.co/caiovicentino1/qwen36-27b-sae-papergrade) via Claude Opus 4.7 (OpenRouter).

**Pipeline** (Bills et al. 2023 / Anthropic auto-interp convention):
1. Stream a corpus through Qwen3.6-27B with hooks at L11 + L31 + L55
2. **Pass A** — count activation frequency per feature → pick the **top-500 per layer** (= 1500 features total)
3. **Pass B** — for each of those 1500 features, capture top-20 activating contexts (32-token windows with the firing token marked)
4. Send each feature's contexts to **Claude Opus 4.7** via OpenRouter — get a short semantic label
5. Save 3× `feature_catalog_L{N}.json` + combined `feature_catalog.json` and upload to the SAE repo

**Cost** (defaults): ~80 min GPU on RTX 6000 Pro 96 GB · ~$25 OpenRouter Opus 4.7 (1500 features × ~750 tok input + ~40 tok output)

**Output**: site will read `feature_catalog.json` and replace `f37602` placeholders with `f37602: cardiovascular anatomy` in the Circuit Canvas, Trace Theater, and Atlas search.

Honest caveats:
- Auto-interp labels are **the LLM's hypothesis** based on 20 contexts — not causally verified ground truth.
- Polysemantic features (firing on multiple unrelated concepts) get explicit `polysemantic: X / Y` labels.
- Early-layer (L11) features often get syntactic labels; late-layer (L31/L55) ones tend to be semantic — this is expected, not a defect.

References: [Bills et al. 2023 (OpenAI auto-interp)](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) · [Templeton et al. 2024 (Scaling Monosemanticity)](https://transformer-circuits.pub/2024/scaling-monosemanticity/)

In [ ]:
# Always latest — Qwen3.6 needs transformers 5.x. requests for OpenRouter.
!pip install -q -U transformers accelerate safetensors huggingface_hub datasets tqdm requests

import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), 'vram:',
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Config

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
LAYERS        = [11, 31, 55]
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

# How many features per layer to label (cost scales linearly).
# 500 per layer × 3 = 1500 total ≈ $25 with Claude Opus 4.7.
TOP_FEATURES_PER_LAYER = 500

# Activation collection budget
CORPUS_TOKENS  = 5_000_000   # 5M tokens — enough firing diversity for the top-500
BATCH_TOKENS   = 4_096        # one batch = 4 sequences of 1024 tokens
SEQ_LEN        = 1_024
CONTEXT_HALF_WIDTH = 16       # 16 tokens before + 16 after the firing token
TOP_CONTEXTS_PER_FEATURE = 20

# OpenRouter — Claude Opus 4.7 by default.
# (User has OPENROUTER_API_KEY in Colab Secrets.)
OPENROUTER_MODEL = 'anthropic/claude-opus-4.7'
OPENROUTER_URL   = 'https://openrouter.ai/api/v1/chat/completions'
API_MAX_TOKENS   = 60          # short labels — keep cost low
API_TEMPERATURE  = 0.0
API_RPM_LIMIT    = 30          # be polite — Opus 4.7 is expensive

import os, math, json, time, random, requests
from collections import defaultdict
random.seed(0); torch.manual_seed(0)

CACHE_DIR = '/content/drive/MyDrive/autointerp_qwen36_27b'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(CACHE_DIR, exist_ok=True)
    print('cache ->', CACHE_DIR)
except Exception:
    CACHE_DIR = '/tmp/autointerp_qwen36_27b'
    os.makedirs(CACHE_DIR, exist_ok=True)

print(f'Total features to label: {TOP_FEATURES_PER_LAYER * len(LAYERS)} '
      f'(top-{TOP_FEATURES_PER_LAYER} per layer × {len(LAYERS)} layers)')

## 2. Auth — HuggingFace + OpenRouter

In [ ]:
from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    assert OPENROUTER_API_KEY, 'OPENROUTER_API_KEY not in Colab Secrets'
except Exception as e:
    login()
    OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY') or input('OPENROUTER_API_KEY: ')

# Smoke test the OpenRouter endpoint with a 1-token prompt
r = requests.post(
    OPENROUTER_URL,
    headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json'},
    json={
        'model': OPENROUTER_MODEL,
        'messages': [{'role': 'user', 'content': 'Reply with exactly: OK'}],
        'max_tokens': 5,
        'temperature': 0.0,
    },
    timeout=30,
)
if r.status_code != 200:
    raise RuntimeError(f'OpenRouter smoke test failed: {r.status_code} {r.text[:300]}')
print('OpenRouter OK ·', OPENROUTER_MODEL, '· reply =', r.json()['choices'][0]['message']['content'].strip())

## 3. Load base model + 3 SAEs (frozen)

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        # x: (..., D_MODEL); returns (..., D_SAE) sparse with k nonzeros along last dim
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z

saes = {}
for layer in LAYERS:
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{layer}_latest.safetensors')
    saes[layer] = TopKSAE(load_file(path), K).to(device).eval()
    print(f'  ✓ SAE L{layer} loaded')

def get_layer_mod(n):
    return model.model.language_model.layers[n]

layer_mods = {layer: get_layer_mod(layer) for layer in LAYERS}
print(f'\nBase frozen + 3 SAEs ready · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 4. Pass A — activation frequency per feature

Single forward sweep over `CORPUS_TOKENS` of FineWeb-Edu, accumulating which features fire how often at each layer. After this we know the **top-500 most-active features per layer**.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

_captured = {}
def capture_hook(key):
    def h(mod, inp, out):
        _captured[key] = out[0] if isinstance(out, tuple) else out
        return out
    return h

def stream_batches(total_tokens=CORPUS_TOKENS, batch_size=BATCH_TOKENS, seq_len=SEQ_LEN):
    """Yield (batch_token_ids, batch_id) packed from FineWeb-Edu."""
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT',
                       split='train', streaming=True)
    buf, seen, batch_id = [], 0, 0
    n_per_batch = batch_size  # tokens
    for ex in ds:
        ids = tok(ex['text'], truncation=True, max_length=seq_len)['input_ids']
        buf.extend(ids)
        while len(buf) >= n_per_batch:
            chunk = buf[:n_per_batch]; buf = buf[n_per_batch:]
            seen += len(chunk)
            B = n_per_batch // seq_len
            arr = torch.tensor(chunk, device=device).reshape(B, seq_len)
            yield arr, batch_id
            batch_id += 1
            if seen >= total_tokens:
                return

# ---- Pass A ----
fire_count = {layer: torch.zeros(D_SAE, dtype=torch.long, device=device) for layer in LAYERS}
max_act    = {layer: torch.zeros(D_SAE, dtype=torch.float32, device=device) for layer in LAYERS}

hs = []
for layer in LAYERS:
    hs.append(layer_mods[layer].register_forward_hook(capture_hook(layer)))

n_steps = CORPUS_TOKENS // BATCH_TOKENS
with torch.no_grad():
    for batch, batch_id in tqdm(stream_batches(), total=n_steps, desc='Pass A · frequency'):
        _ = model(batch)
        for layer in LAYERS:
            resid = _captured[layer]                  # (B, T, D_MODEL)
            B_, T_, _ = resid.shape
            flat = resid.reshape(-1, D_MODEL)         # (B*T, D)
            z = saes[layer].encode(flat.to(torch.bfloat16))  # (B*T, D_SAE) sparse
            fire = (z > 0).sum(dim=0)                 # (D_SAE,)
            fire_count[layer] += fire
            cur_max = z.max(dim=0).values.float()     # (D_SAE,)
            max_act[layer] = torch.maximum(max_act[layer], cur_max)

for h in hs:
    h.remove()

# Pick top features per layer
top_features = {}
for layer in LAYERS:
    top = torch.topk(fire_count[layer], TOP_FEATURES_PER_LAYER).indices.cpu().tolist()
    top_features[layer] = top
    print(f'  L{layer}: top-{TOP_FEATURES_PER_LAYER} fire-counts: '
          f'min={fire_count[layer][top].min().item()} '
          f'max={fire_count[layer][top].max().item()} '
          f'max_act_avg={max_act[layer][top].mean().item():.2f}')

# Checkpoint
torch.save({
    'fire_count': {l: v.cpu() for l, v in fire_count.items()},
    'max_act':    {l: v.cpu() for l, v in max_act.items()},
    'top_features': top_features,
}, os.path.join(CACHE_DIR, 'pass_a.pt'))
print('  ✓ pass A saved')

## 5. Pass B — top-20 activating contexts per top-feature

Re-stream the corpus, capture context windows around the top activation positions for each of the 1500 selected features.

In [ ]:
import heapq

# (-act_val, batch_id, position, full_token_window) — neg act so heapq pops the smallest = least-strong
# We keep TOP_CONTEXTS_PER_FEATURE strongest by maintaining a min-heap of size K.
ctx_heap = {layer: {f: [] for f in top_features[layer]} for layer in LAYERS}
top_set  = {layer: set(top_features[layer]) for layer in LAYERS}

hs = []
for layer in LAYERS:
    hs.append(layer_mods[layer].register_forward_hook(capture_hook(layer)))

with torch.no_grad():
    for batch, batch_id in tqdm(stream_batches(), total=n_steps, desc='Pass B · contexts'):
        _ = model(batch)
        B_, T_ = batch.shape
        for layer in LAYERS:
            resid = _captured[layer]
            flat = resid.reshape(-1, D_MODEL)
            z = saes[layer].encode(flat.to(torch.bfloat16))   # (B*T, D_SAE)
            z = z.view(B_, T_, D_SAE)
            # For each tracked feature, find any nonzero positions in this batch
            tracked = top_features[layer]
            sub_z = z[:, :, tracked]   # (B, T, len_tracked)
            # Find the (b, t, idx_in_tracked) for nonzero entries
            mask = sub_z > 0
            if not mask.any():
                continue
            nz_b, nz_t, nz_i = mask.nonzero(as_tuple=True)
            vals = sub_z[nz_b, nz_t, nz_i].float().cpu().tolist()
            nz_b = nz_b.cpu().tolist(); nz_t = nz_t.cpu().tolist(); nz_i = nz_i.cpu().tolist()
            batch_cpu = batch.cpu()
            for v, b, t, i in zip(vals, nz_b, nz_t, nz_i):
                feat = tracked[i]
                heap = ctx_heap[layer][feat]
                if len(heap) < TOP_CONTEXTS_PER_FEATURE:
                    heapq.heappush(heap, (v, batch_id, b, t))
                elif v > heap[0][0]:
                    heapq.heapreplace(heap, (v, batch_id, b, t))
        # Cache batch tokens for context extraction (only most-recent N batches kept on disk)
        torch.save(batch.cpu(), os.path.join(CACHE_DIR, f'batch_{batch_id:06d}.pt'))

for h in hs:
    h.remove()

# Reify contexts: replace (batch_id, b, t) with the actual token-window text
def reify_window(batch_id, b, t):
    arr = torch.load(os.path.join(CACHE_DIR, f'batch_{batch_id:06d}.pt'))
    seq = arr[b].tolist()
    lo = max(0, t - CONTEXT_HALF_WIDTH)
    hi = min(len(seq), t + CONTEXT_HALF_WIDTH + 1)
    pre  = tok.decode(seq[lo:t],   skip_special_tokens=True)
    fire = tok.decode([seq[t]],    skip_special_tokens=True)
    post = tok.decode(seq[t+1:hi], skip_special_tokens=True)
    return pre, fire, post

contexts_by_feature = {layer: {} for layer in LAYERS}
for layer in LAYERS:
    for feat, heap in tqdm(list(ctx_heap[layer].items()), desc=f'L{layer} reify'):
        # heap entries: (v, batch_id, b, t) — keep all
        items = sorted(heap, key=lambda x: -x[0])  # strongest first
        triples = [(reify_window(batch_id, b, t), v) for v, batch_id, b, t in items]
        contexts_by_feature[layer][feat] = [
            {'pre': p, 'fire': f, 'post': po, 'act': float(v)}
            for ((p, f, po), v) in triples
        ]

# Save + free batch caches
with open(os.path.join(CACHE_DIR, 'contexts.json'), 'w') as f:
    json.dump({str(l): {str(k): v for k, v in d.items()} for l, d in contexts_by_feature.items()}, f, indent=2)
import glob
for p in glob.glob(os.path.join(CACHE_DIR, 'batch_*.pt')):
    os.remove(p)
print(f'  ✓ contexts saved · removed batch cache')

## 6. Auto-interp — Claude Opus 4.7 via OpenRouter

For each feature, send the top-20 contexts to Claude with a structured prompt. Use deterministic temperature 0 for reproducibility. Built-in retry + RPM throttle.

In [ ]:
PROMPT_TEMPLATE = '''You are analyzing a feature in a Sparse Autoencoder trained on the residual stream of Qwen3.6-27B at layer {layer}.

Below are {n} token contexts where this feature activates strongly. Each context shows a window of text around the firing token. The token where the feature actually fires is marked with [[double brackets]].

{contexts_block}

Based on these contexts, what concept does this feature detect? Look for syntactic patterns, semantic categories, named entities, factual associations, or stylistic features. Respond with a SHORT label (3-8 words) that captures what unifies the activations.

If the feature appears polysemantic (firing on multiple unrelated concepts), respond with "polysemantic: X / Y" naming the two strongest patterns.

Reply with ONLY the label — no quotes, no preamble, no period.
'''

def format_contexts(ctxs):
    lines = []
    for i, c in enumerate(ctxs, 1):
        text = (c['pre'] + '[[' + c['fire'] + ']]' + c['post']).replace('\n', ' ')
        lines.append(f'{i}. "{text.strip()}"  (act={c["act"]:.2f})')
    return '\n'.join(lines)

def call_claude(prompt, retries=4):
    for attempt in range(retries):
        try:
            r = requests.post(
                OPENROUTER_URL,
                headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json'},
                json={
                    'model': OPENROUTER_MODEL,
                    'messages': [{'role': 'user', 'content': prompt}],
                    'max_tokens': API_MAX_TOKENS,
                    'temperature': API_TEMPERATURE,
                },
                timeout=60,
            )
            if r.status_code == 200:
                data = r.json()
                label = data['choices'][0]['message']['content'].strip().strip('"').strip("'").strip('.')
                usage = data.get('usage', {})
                return label, usage
            elif r.status_code == 429:
                wait = 2 ** attempt * 5
                print(f'    rate-limited; sleeping {wait}s')
                time.sleep(wait)
            else:
                print(f'    http {r.status_code}: {r.text[:200]}')
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f'    exception {e}; retry {attempt}')
            time.sleep(2 ** attempt)
    return None, {}

# Resume support — load any prior progress
labels = {layer: {} for layer in LAYERS}
labels_path = os.path.join(CACHE_DIR, 'labels_partial.json')
if os.path.exists(labels_path):
    with open(labels_path) as f:
        prior = json.load(f)
    for layer in LAYERS:
        labels[layer] = {int(k): v for k, v in prior.get(str(layer), {}).items()}
    total_done = sum(len(v) for v in labels.values())
    print(f'  resumed from {total_done} prior labels')

min_request_dt = 60.0 / API_RPM_LIMIT
last_request = 0.0
total_in_tokens = total_out_tokens = 0

for layer in LAYERS:
    feats = top_features[layer]
    pending = [f for f in feats if f not in labels[layer]]
    print(f'\n=== L{layer} · labeling {len(pending)} features ===')
    for i, feat in enumerate(tqdm(pending, desc=f'L{layer}')):
        ctxs = contexts_by_feature[layer][feat]
        if not ctxs:
            labels[layer][feat] = '(no activations)'
            continue
        prompt = PROMPT_TEMPLATE.format(
            layer=layer,
            n=len(ctxs),
            contexts_block=format_contexts(ctxs),
        )
        # Throttle
        dt = time.time() - last_request
        if dt < min_request_dt:
            time.sleep(min_request_dt - dt)
        last_request = time.time()

        label, usage = call_claude(prompt)
        if label is None:
            label = '(api failed)'
        labels[layer][feat] = label
        total_in_tokens  += usage.get('prompt_tokens', 0)
        total_out_tokens += usage.get('completion_tokens', 0)

        # Save partial every 25 labels — survives Colab kills
        if (i + 1) % 25 == 0:
            with open(labels_path, 'w') as f:
                json.dump({str(l): {str(k): v for k, v in d.items()} for l, d in labels.items()}, f)

    # Layer done — save
    with open(labels_path, 'w') as f:
        json.dump({str(l): {str(k): v for k, v in d.items()} for l, d in labels.items()}, f)

# Cost report (rough — Opus 4.7 list price)
in_cost  = total_in_tokens  / 1_000_000 * 15.0
out_cost = total_out_tokens / 1_000_000 * 75.0
print(f'\n  total tokens · in={total_in_tokens:,} · out={total_out_tokens:,}')
print(f'  estimated cost · ${in_cost + out_cost:.2f} '
      f'(in ${in_cost:.2f} + out ${out_cost:.2f}) at Opus 4.7 list price')

## 7. Build feature_catalog.json + per-layer files

In [ ]:
from datetime import datetime, timezone

for layer in LAYERS:
    catalog = {
        'version': 'v0.1.0',
        'model':   HF_BASE_MODEL,
        'sae_repo': HF_SAE_REPO,
        'layer':   layer,
        'n_labeled': len(labels[layer]),
        'top_features_per_layer': TOP_FEATURES_PER_LAYER,
        'corpus_tokens': CORPUS_TOKENS,
        'top_contexts_per_feature': TOP_CONTEXTS_PER_FEATURE,
        'auto_interp_model': OPENROUTER_MODEL,
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'labels': {},   # str(feature_id) -> {label, fire_count, max_act, n_contexts}
    }
    for feat, label in labels[layer].items():
        catalog['labels'][str(feat)] = {
            'label': label,
            'fire_count': int(fire_count[layer][feat].item()),
            'max_act':    float(max_act[layer][feat].item()),
            'n_contexts': len(contexts_by_feature[layer].get(feat, [])),
        }
    out_path = os.path.join(CACHE_DIR, f'feature_catalog_L{layer}.json')
    with open(out_path, 'w') as f:
        json.dump(catalog, f, indent=2)
    print(f'  ✓ L{layer}: {len(catalog["labels"])} labels → {out_path}')

# Combined index
combined = {
    'version':  'v0.1.0',
    'model':    HF_BASE_MODEL,
    'sae_repo': HF_SAE_REPO,
    'layers':   LAYERS,
    'auto_interp_model': OPENROUTER_MODEL,
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'per_layer_files': [f'feature_catalog_L{l}.json' for l in LAYERS],
    'labels': {
        str(layer): {
            str(feat): label
            for feat, label in labels[layer].items()
        }
        for layer in LAYERS
    },
}
combined_path = os.path.join(CACHE_DIR, 'feature_catalog.json')
with open(combined_path, 'w') as f:
    json.dump(combined, f, indent=2)
print(f'\n  combined: {combined_path}')

## 8. Upload catalogs to HF SAE repo

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

for layer in LAYERS:
    api.upload_file(
        path_or_fileobj=os.path.join(CACHE_DIR, f'feature_catalog_L{layer}.json'),
        path_in_repo=f'feature_catalog_L{layer}.json',
        repo_id=HF_SAE_REPO,
        commit_message=f'Auto-interp · L{layer} · {len(labels[layer])} labels (Claude Opus 4.7)',
    )
    print(f'  ✓ feature_catalog_L{layer}.json')

api.upload_file(
    path_or_fileobj=os.path.join(CACHE_DIR, 'feature_catalog.json'),
    path_in_repo='feature_catalog.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'Auto-interp · combined catalog · {sum(len(v) for v in labels.values())} labels',
)
print(f'\n✓ all catalogs uploaded → https://huggingface.co/{HF_SAE_REPO}/tree/main')
print(f'\nNext: site reads feature_catalog.json and merges labels into Circuit Canvas + Trace Theater + Atlas.')

## 9. Spot-check — show 30 random labels per layer

Quick sanity. Look for: (1) coherent semantic clusters, (2) reasonable polysemantic flags, (3) early-layer should skew syntactic, late-layer should skew semantic.

In [ ]:
import random as _r
for layer in LAYERS:
    print(f'\n=== L{layer} sample ===')
    sample = _r.Random(0).sample(list(labels[layer].items()), min(30, len(labels[layer])))
    for feat, label in sample:
        print(f'  L{layer}/f{feat:5d}  fc={int(fire_count[layer][feat]):>5d}  {label}')